In [83]:
from transformers import CLIPProcessor, CLIPModel, CLIPTokenizer
import torch
import pandas as pd
import numpy as np
from PIL import Image
import os
from tqdm.auto import tqdm

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [4]:
model_id = "openai/clip-vit-large-patch14"
model = CLIPModel.from_pretrained(model_id).to(device)
processor = CLIPProcessor.from_pretrained(model_id)
tokenizer = CLIPTokenizer.from_pretrained(model_id)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

In [94]:
df = pd.read_csv("metadata.csv")

df = df.dropna(subset=["Image File", "Category"]) # drop rows with missing values
# use the first 100, can be changed for other subsets
df = df[:1000]
true_labels = list(df["Category"])
categories = list(df["Category"].unique())

image_paths = [
    os.path.join("images2", fname) # change images2 to name of folder with images
    for fname in df["Image File"]
]

In [72]:
# convert the category labels into text inputs using the tokenizer
text_inputs = tokenizer([f"a photo of a {c}" for c in categories], return_tensors="pt", padding=True).to(device)

In [73]:
# encode tokens to sentence embeddings
with torch.no_grad():
    label_emb = model.get_text_features(input_ids=text_inputs['input_ids'], attention_mask=text_inputs['attention_mask'])
    label_emb = label_emb.cpu().numpy()

In [74]:
label_emb.min(), label_emb.max() 

(np.float32(-11.811214), np.float32(10.146707))

In [75]:
# normalization for doing dot product similarity
label_emb = label_emb/np.linalg.norm(label_emb,axis=0)
label_emb.min(), label_emb.max()

(np.float32(-0.8197615), np.float32(0.8266128))

In [76]:
def get_image_embedding(image_path: str):
    image = Image.open(image_path).convert("RGB")
    with torch.no_grad():
        inputs = processor(images=image, return_tensors="pt", padding=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        return model.get_image_features(**inputs)

image_emb = torch.cat([get_image_embedding(path) for path in image_paths])
image_emb = image_emb.cpu().numpy()

KeyboardInterrupt: 

In [107]:
preds = []
batch_size = 32
all_image_emb = []

for i in  tqdm(range(0, len(image_paths), batch_size)):
    i_end = min(i + batch_size, len(image_paths))
    
    # load images
    batch_images = [Image.open(p).convert("RGB") for p in image_paths[i:i_end]]
    with torch.no_grad():
        inputs = processor(images=batch_images, return_tensors="pt", padding=True).to(device)
        batch_emb = model.get_image_features(**inputs)
        
        # normalize 
        batch_emb = batch_emb / np.linalg.norm(batch_emb,axis=0)
    all_image_emb.append(batch_emb.cpu().numpy())

image_emb = np.vstack(all_image_emb) 


  0%|          | 0/32 [00:00<?, ?it/s]

In [108]:
scores = np.dot(image_emb, label_emb.T)
preds.extend(np.argmax(scores, axis=1))

In [109]:
categories[preds[2]]

'Elektronik'

In [111]:
label_to_idx = {label: i for i, label in enumerate(categories)}
true_idx = np.array([label_to_idx[l] for l in true_labels])

In [115]:
# preds: shape (1000,) – predicted label indices
# true_idx: shape (1000,) – true label indices

# calculate accuracy (percentage of correct predictions)
true_preds = []
for i, label in enumerate(true_idx):
    if label == preds[i]:
        true_preds.append(1)
    else:
        true_preds.append(0)
        
sum(true_preds) /len(true_preds) 

0.117

In [113]:
for i in range(10):
    print(
        "GT:", true_labels[i],
        "| Pred:", categories[preds[i]]
    )

GT: Fritid & Have | Pred: Byggematerialer
GT: Lamper | Pred: Legetøj
GT: Elektronik | Pred: Elektronik
GT: Fritid & Have | Pred: Byggematerialer
GT: Boligting | Pred: Byggematerialer
GT: Boligting | Pred: Møbler
GT: Fritid & Have | Pred: Byggematerialer
GT: Musik & Bøger | Pred: Byggematerialer
GT: Lamper | Pred: Elektronik
GT: Musik & Bøger | Pred: Byggematerialer


In [117]:
top5 = np.argsort(scores, axis=1)[:, -5:]
top5_acc = np.mean([true_idx[i] in top5[i] for i in range(len(true_idx))])
print("Top-5 accuracy:", top5_acc)

Top-5 accuracy: 0.519
